In [ ]:
!pip install requests beautifulsoup4 lxml -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import pandas as pd
from datetime import datetime
import urllib3
import urllib.parse
import re


In [ ]:
GG_COLAB = "/content/drive/MyDrive/"
SAVE_URL_PATH = f'{GG_COLAB}/laborlaw/collected_urls'
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [ ]:

class CourtDecisionScraper:
    def __init__(self):
        self.base_url = "https://congbobanan.toaan.gov.vn"
        self.search_url = f"{self.base_url}/0tat1cvn/ban-an-quyet-dinh"
        self.results = []
        self.last_viewstate = {}

        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
            'Accept-Language': 'vi-VN,vi;q=0.9,en;q=0.8',
            'Referer': self.search_url,
            'Origin': self.base_url,
            'Content-Type': 'application/x-www-form-urlencoded',
        })

    def get_initial_page(self):
        #GET trang đầu để lấy ViewState
        # print("Lấy ViewState từ trang đầu...")
        response = self.session.get(self.search_url, verify=False, timeout=60)

        if response.status_code != 200:
            raise Exception(f"Không thể GET trang: {response.status_code}")

        soup = BeautifulSoup(response.text, 'lxml')

        viewstate = soup.find('input', {'name': '__VIEWSTATE'})
        viewstate_value = viewstate['value'] if viewstate else ''

        eventvalidation = soup.find('input', {'name': '__EVENTVALIDATION'})
        eventvalidation_value = eventvalidation['value'] if eventvalidation else ''

        viewstategenerator = soup.find('input', {'name': '__VIEWSTATEGENERATOR'})
        viewstategenerator_value = viewstategenerator['value'] if viewstategenerator else ''

        # print(f"ViewState length: {len(viewstate_value)}")
        # print(f"EventValidation length: {len(eventvalidation_value)}")

        return {
            '__VIEWSTATE': viewstate_value,
            '__EVENTVALIDATION': eventvalidation_value,
            '__VIEWSTATEGENERATOR': viewstategenerator_value
        }
    def get_total_pages(self, html):
        #Lấy số trang từ dropdown pagination
        soup = BeautifulSoup(html, 'lxml')
        dropdown = soup.find('select', {'name': 'ctl00$Content_home_Public$ctl00$DropPages'})

        if dropdown:
            options = dropdown.find_all('option')
            total = len(options)
            print(f"--->Tổng số trang có sẵn: {total}")
            return total
        return None
    def search_labor_decisions(self, page=1):

       # Tìm kiếm bản án lao động


        print(f"TRANG {page}")
        # Lấy ViewState
        if page == 1:
            viewstate_data = self.get_initial_page()
        else:
            viewstate_data = self.last_viewstate

        # Build POST data
        form_data = {
            'ctl00$Feedback_Home$hdnProcess': 'FALSE',
            'ctl00$Feedback_Home$Radio_STYLE': '13',

            # Search fields phía trên
            'ctl00$Content_home_Public$ctl00$txtKeyword_top': '',
            'ctl00$Content_home_Public$ctl00$Drop_Levels_top': '',
            'ctl00$Content_home_Public$ctl00$Ra_Drop_Courts_top': '',
            'ctl00$Content_home_Public$ctl00$Drop_STATUS_JUDGMENT_SEARCH_top': '0',
            'ctl00$Content_home_Public$ctl00$Drop_CASES_STYLES_SEARCH_top': '3',
            'ctl00$Content_home_Public$ctl00$Ra_Case_shows_search_top': '',
            'ctl00$Content_home_Public$ctl00$Rad_DATE_FROM_top': '01/01/2020',
            'ctl00$Content_home_Public$ctl00$MaskedEditExtender1_ClientState': '',
            'ctl00$Content_home_Public$ctl00$Rad_DATE_TO_top': '31/12/2026',
            'ctl00$Content_home_Public$ctl00$MaskedEditExtender4_ClientState': '',

            # Search fields phía dưới
            'ctl00$Content_home_Public$ctl00$txtKeyword': '',
            'ctl00$Content_home_Public$ctl00$Drop_Levels': '',
            'ctl00$Content_home_Public$ctl00$Ra_Drop_Courts': '',
            'ctl00$Content_home_Public$ctl00$Drop_STATUS_JUDGMENT_SEARCH': '0',
            'ctl00$Content_home_Public$ctl00$Drop_CASES_STYLES_SEARCH': '3',
            'ctl00$Content_home_Public$ctl00$Ra_Case_shows_search': '',
            'ctl00$Content_home_Public$ctl00$Rad_DATE_FROM': '01/01/2020',
            'ctl00$Content_home_Public$ctl00$MaskedEditExtender2_ClientState': '',
            'ctl00$Content_home_Public$ctl00$Rad_DATE_TO': '31/12/2026',
            'ctl00$Content_home_Public$ctl00$MaskedEditExtender3_ClientState': '',

            # Pagination
            'ctl00$Content_home_Public$ctl00$DropPages': str(page),
            'ctl00$Content_home_Public$ctl00$Hi_id_pub': '',

            # CHỈ SET __EVENTTARGET KHI CHUYỂN TRANG (page > 1)
            '__EVENTTARGET': 'ctl00$Content_home_Public$ctl00$DropPages' if page > 1 else '',
            '__EVENTARGUMENT': '',
            '__LASTFOCUS': '',
        }

        # TRANG 1 CẦN NÚT SEARCH
        if page == 1:
            form_data['ctl00$Content_home_Public$ctl00$cmd_search_banner'] = 'Tìm kiếm'

        # Thêm ViewState
        form_data.update(viewstate_data)

        # POST request
        response = self.session.post(
            self.search_url,
            data=form_data,
            verify=False,
            timeout=60
        )

        # print(f"Response status: {response.status_code}")
        # print(f"Response length: {len(response.text)} bytes")

        if response.status_code != 200:
            print(f"Lỗi POST: {response.status_code}")
            return []

        # Cập nhật ViewState
        soup = BeautifulSoup(response.text, 'lxml')
        viewstate_input = soup.find('input', {'name': '__VIEWSTATE'})
        eventvalidation_input = soup.find('input', {'name': '__EVENTVALIDATION'})
        viewstategenerator_input = soup.find('input', {'name': '__VIEWSTATEGENERATOR'})

        self.last_viewstate = {
            '__VIEWSTATE': viewstate_input['value'] if viewstate_input else '',
            '__EVENTVALIDATION': eventvalidation_input['value'] if eventvalidation_input else '',
            '__VIEWSTATEGENERATOR': viewstategenerator_input['value'] if viewstategenerator_input else ''
        }
        # Lấy tổng số trang (chỉ làm ở trang 1)
        total_pages = None
        if page == 1:
            total_pages = self.get_total_pages(response.text)


        # Parse kết quả
        links = self.extract_links(response.text)
        print(f"Tìm thấy {len(links)} links")

        return links,total_pages


    def extract_links(self, html):
        #Trích xuất links bản án từ HTML
        soup = BeautifulSoup(html, 'lxml')
        links = []

        print("Tìm links trong HTML")

        # Tìm tất cả links
        all_links = soup.find_all('a', href=True)
        print(f"==>Tổng số <a> tags: {len(all_links)}")

        for a in all_links:
            href = a['href']
            text = a.get_text(strip=True)

            if len(links) < 3:
                print(f"===>link: {href[:80]}... =>Text: {text[:50]}")

            # mở rộng filter
            if any(keyword in href.lower() for keyword in [
                'chi-tiet-ban-an',  # Pattern chính
                'chi-tiet',
                'detail',
                '/2ta',  # Pattern trong URL
                'id='
            ]):
                # Build full URL
                if href.startswith('/'):
                    full_url = f"{self.base_url}{href}"
                elif href.startswith('http'):
                    full_url = href
                else:
                    full_url = f"{self.base_url}/{href}"

                links.append({
                    'url': full_url,
                    'title': text,
                    'href': href
                })

        print(f"=> Filtered: {len(links)} links phù hợp")
        return links


    def scrape_all_pages(self, max_pages=None):
        #Scrape nhiều trang
        all_links = []
        page = 1
        total_pages = None
        seen_urls = set() # check url duplicate

        while True:
            if max_pages and page > max_pages:
                print(f"Đã đạt giới hạn {max_pages} trang")
                break
            if total_pages and page > total_pages:
                print(f'Đã đạt trang cuối cùng ({total_pages})"')
            # thu thập cho trang hiện tại
            links,detected_total = self.search_labor_decisions(page=page)
            # Lưu total_pages từ trang đầu
            if detected_total and not total_pages:
                total_pages = detected_total
            if not links:
                print(f"Ko có links ở trang {page}, dừng")
                break
            # phát hiện duplicate, dấu hiệu đã hết trang
            new_links = 0
            duplicate_count = 0
            for link in links:
                if link['url'] not in seen_urls:
                    all_links.append(link)
                    seen_urls.add(link['url'])
                    new_links += 1
                else:
                    duplicate_count+=1

            print(f"--> Check url trùng: Mới: {new_links} | Trùng: {duplicate_count}")

            # all link trùng => hết trang real
            if new_links == 0 and duplicate_count > 0:
                print(f'Tất cả links ở trang {page} đều trùng với trang trước')
                print('Hết trang ==> DỪNG')
                break

            print(f"Tổng cộng: {len(all_links)} links")

            page += 1


            import time
            time.sleep(5) # 5 giay

        print(f"==>Đã crawl: {page - 1} trang")
        print(f"==>Tổng: {len(all_links)} URL duy nhất")
        self.results = all_links
        return all_links

    def save_results(self, filename='url_ban_an_lao_dong'):
        #Lưu kết quả
        if not self.results:
            print("Không có dữ liệu để lưu!")
            return

        # JSON
        with open(f'{SAVE_URL_PATH}/{filename}.json', 'w', encoding='utf-8') as f:
            json.dump(self.results, f, ensure_ascii=False, indent=2)
        print(f"{SAVE_URL_PATH}/{filename}.json")

        # CSV
        pd.DataFrame(self.results).to_csv(f'{SAVE_URL_PATH}/{filename}.csv', index=False, encoding='utf-8-sig')
        print(f"{SAVE_URL_PATH}/{filename}.csv")

        # TXT
        with open(f'{SAVE_URL_PATH}/{filename}_urls.txt', 'w', encoding='utf-8') as f:
            for item in self.results:
                f.write(item['url'] + '\n')
        print(f"{SAVE_URL_PATH}/{filename}_urls.txt")


scraper = CourtDecisionScraper()

#results = scraper.scrape_all_pages(max_pages=3) # limit
results = scraper.scrape_all_pages()

# Hiển thị mẫu

print("KẾT QUẢ:")
for i, item in enumerate(results, 1):
    print(f"{i}. {item['url']}")
    if item['title']:
        print(f"==> {item['title']}")

# Lưu kết quả
print("LƯU KẾT QUẢ:")

scraper.save_results('ban_an_lao_dong')


Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
1206. https://congbobanan.toaan.gov.vn/2ta1715238t1cvn/chi-tiet-ban-an
==> Bản án:số 22 ngày 30/12/2024 của TAND TP Đà Nẵng(07.02.2025)
1207. https://congbobanan.toaan.gov.vn/2ta1715244t1cvn/chi-tiet-ban-an
==> Bản án:số 20 ngày 30/12/2024 của TAND TP Đà Nẵng(07.02.2025)
1208. https://congbobanan.toaan.gov.vn/2ta1715227t1cvn/chi-tiet-ban-an
==> Bản án:số 18 ngày 30/12/2024 của TAND TP Đà Nẵng(07.02.2025)
1209. https://congbobanan.toaan.gov.vn/2ta1714876t1cvn/chi-tiet-ban-an
==> Bản án:số 03/2024/LĐPT ngày 26/12/2024 của TAND tỉnh Tiền Giang(06.02.2025)
1210. https://congbobanan.toaan.gov.vn/2ta1714541t1cvn/chi-tiet-ban-an
==> Bản án:số 01 ngày 10/01/2025 của TAND tỉnh Bạc Liêu(06.02.2025)
1211. https://congbobanan.toaan.gov.vn/2ta1711966t1cvn/chi-tiet-ban-an
==> Bản án:số 22/2025/LĐ-PT ngày 16/01/2025 của TAND TP Đà Nẵng(03.02.2025)
1212. https://congbobanan.toaan.gov.vn/2ta1711973t1cvn/chi-tiet-ban-an
==> Bản án:số 26/2025/LĐ-PT